# Phase 10: Report-ready evidence views

This notebook reads the saved Phase 4–8 artifacts and formats genuine, screenshot-ready evidence for the final report. It does not fit a model, select a model, alter the locked split, or simulate new results. The model chapters use the family-selected sigmoid variants (KNN, Gaussian Naive Bayes, and logistic regression), matching the report.

Run the notebook from VS Code with the project `.venv` kernel. Before capturing, use **Restart Kernel and Run All** so all displayed artifacts come from the saved files. Capture the visible notebook cells and outputs directly. You may crop a genuine UI capture to focus on the relevant cell and result, but do not change values or recreate the interface.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path.cwd().resolve()
for parent in [candidate, *candidate.parents]:
    if (parent / '.venv').is_dir() and (parent / 'README.md').is_file() and (parent / 'reports').is_dir():
        PROJECT_ROOT = parent
        break
else:
    raise RuntimeError('Open this notebook inside the project folder so its .venv and README.md can be located.')

assert Path(__import__('sys').prefix).resolve() == (PROJECT_ROOT / '.venv').resolve(), 'Select the project .venv kernel in VS Code.'
TABLES = PROJECT_ROOT / 'reports' / 'tables'
DOCS = PROJECT_ROOT / 'docs'
print('Project:', PROJECT_ROOT)
print('Python kernel:', __import__('sys').executable)
print('All views below load saved results only.')


## Dataset and preprocessing evidence

The source table is shown only as an excerpt. The label checks and reconciliation counts are loaded from the saved Phase 4 report; they are not recalculated by this evidence notebook.


In [ ]:
shots = pd.read_csv(PROJECT_ROOT / 'data/processed/shots.csv', nrows=8)
phase4 = json.loads((DOCS / 'shot_dataset_report.json').read_text(encoding='utf-8'))
display(shots[['match_id','event_id','team_name','shot_outcome','distance_sb_units','angle_sb_radians','is_goal']].head(8))
display(pd.DataFrame([
    ('Shots', phase4['eligible_shot_count']),
    ('Goals', phase4['goal_count']),
    ('Non-goals', phase4['non_goal_count']),
    ('All match scores reconciled', phase4['all_matches_reconciled']),
], columns=['Quality check','Saved result']))


Capture these as `preprocessing_01_dataset.png`, `preprocessing_02_labels_features.png`, and `preprocessing_03_quality_reconciliation.png`. For the latter two, scroll to the Phase 4 builder notebook cells that actually construct labels/features and validate own-goal reconciliation; this evidence notebook intentionally displays saved summaries only.

## Split and grouped-fold evidence

These saved maps establish the match-group split and ten-fold summary. The group assertion below checks that the persisted train and test match identifiers are disjoint.


In [ ]:
membership = pd.read_csv(PROJECT_ROOT / 'data/processed/split_membership.csv')
split_report = json.loads((DOCS / 'phase_05_eda_report.json').read_text(encoding='utf-8'))
cv = pd.read_csv(TABLES / 'phase06_cv_summary.csv')
train_ids = set(membership.loc[membership.split == 'train', 'match_id'])
test_ids = set(membership.loc[membership.split == 'test', 'match_id'])
assert train_ids.isdisjoint(test_ids)
print(f"Train: {len(train_ids)} matches; {membership.loc[membership.split == 'train','shot_count'].sum():,} shots")
print(f"Test: {len(test_ids)} matches; {membership.loc[membership.split == 'test','shot_count'].sum():,} shots")
print(f"Group overlap: {len(train_ids & test_ids)} matches")
display(cv[['fold','validation_match_count','validation_shot_count','validation_goal_count']])


Capture the allocation, group separation, and ten-fold output as `split_01_match_allocation.png`, `split_02_group_separation.png`, and `split_03_ten_fold_cv.png`.

## Model evidence: scope definitions

Training resubstitution scores each fitted final training pipeline on rows used to fit it and is therefore optimistic. Ten-fold pooled OOF metrics use one prediction per training shot from a fold model that did not train on its match; the fold table reports fold-to-fold standard deviation, not a confidence interval. Test metrics use the frozen 1,834-shot held-out partition. All metrics below are loaded from CSV; no models are fitted.


In [ ]:
training = pd.read_csv(TABLES / 'phase08_training_metrics.csv')
development = pd.read_csv(TABLES / 'phase08_development_metrics.csv')
folds = pd.read_csv(TABLES / 'phase08_fold_metrics.csv')
testing = pd.read_csv(TABLES / 'phase08_test_metrics.csv')
selected = {
    'KNN 50 neighbours (sigmoid selected)': 'knn_50_sigmoid',
    'Gaussian Naive Bayes (sigmoid selected)': 'gaussian_nb_sigmoid',
    'Logistic regression (sigmoid selected)': 'logistic_regression_sigmoid',
}
def scope_table(variant):
    parts=[]
    for label, frame in [('Training resubstitution',training),('10-fold pooled OOF',development),('Held-out test',testing)]:
        row=frame.loc[frame.variant.eq(variant)].iloc[0]
        parts.append({'Scope':label,'Shots':int(row.shot_count),'Goals':int(row.goal_count),
                      'Accuracy':row.accuracy,'Precision':row.precision,'Recall':row.recall,
                      'F1':row.f1,'Log loss':row.log_loss,'Brier':row.brier_score})
    return pd.DataFrame(parts)


### KNN, 50 neighbours — sigmoid-calibrated family-selected variant

The family-selected variant is the sigmoid-calibrated version chosen by lower pooled training-only OOF log loss than its raw counterpart. These are the three model chapters reported below; the logistic-regression sigmoid variant is the pre-frozen global candidate.


In [ ]:
variant = 'knn_50_sigmoid'
print('KNN, 50 neighbours: same fixed 0.5 decision threshold for hard predictions')
display(scope_table(variant).round(4))
fold_view = folds.loc[folds.variant.eq(variant), ['fold','validation_shot_count','validation_goal_count','log_loss','brier_score','accuracy','precision','recall','f1']].copy()
print('Ten validation folds; standard deviation below describes fold variation, not a confidence interval.')
display(fold_view.round(4))
print('Unweighted fold mean and sample standard deviation (ddof=1):')
display(fold_view.drop(columns=['fold']).agg(['mean','std']).round(4))


In [ ]:
row = testing.loc[testing.variant.eq('knn_50_sigmoid')].iloc[0]
print(f"Held-out test: {int(row.shot_count):,} shots, {int(row.goal_count):,} goals; fixed threshold = {row.threshold:.1f}")
print('Probability metrics:')
display(pd.DataFrame([{'Log loss':row.log_loss,'Brier':row.brier_score,
                        'Average precision':row.average_precision,'ROC-AUC':row.roc_auc}]).round(4))
print('Classification metrics and threshold counts:')
display(pd.DataFrame([{'Accuracy':row.accuracy,'Precision':row.precision,'Recall':row.recall,
                        'F1':row.f1,'TN':int(row.tn),'FP':int(row.fp),'FN':int(row.fn),'TP':int(row.tp)}]).round(4))
fig, ax = plt.subplots(figsize=(4.5, 3.5))
matrix = [[int(row.tn), int(row.fp)], [int(row.fn), int(row.tp)]]
im = ax.imshow(matrix, cmap='Blues')
for i in range(2):
    for j in range(2):
        text_color = 'white' if matrix[i][j] > max(max(values) for values in matrix) / 2 else 'black'
        ax.text(j, i, f'{matrix[i][j]:,}', ha='center', va='center', color=text_color, fontsize=12, fontweight='bold')
ax.set_xticks([0,1], ['Non-goal','Goal'])
ax.set_yticks([0,1], ['Non-goal','Goal'])
ax.set_title('KNN, 50 neighbours: held-out test confusion matrix')
ax.set_xlabel('Predicted class'); ax.set_ylabel('Actual class')
fig.text(.5, .01, 'Test only · threshold 0.5 · StatsBomb Open Data', ha='center', fontsize=8)
fig.tight_layout(rect=[0,.05,1,1]); plt.show()


Capture training, CV, and test evidence as `knn_01_training.png`, `knn_02_cross_validation.png`, and `knn_03_test.png`. Capture the held-out metrics table and confusion matrix separately as `evaluation_knn_test_metrics.png` and `evaluation_knn_test_confusion.png`. Preserve the visible title and scope label in every image.

### Gaussian Naive Bayes — sigmoid-calibrated family-selected variant

The family-selected variant is the sigmoid-calibrated version chosen by lower pooled training-only OOF log loss than its raw counterpart. These are the three model chapters reported below; the logistic-regression sigmoid variant is the pre-frozen global candidate.


In [ ]:
variant = 'gaussian_nb_sigmoid'
print('Gaussian Naive Bayes: same fixed 0.5 decision threshold for hard predictions')
display(scope_table(variant).round(4))
fold_view = folds.loc[folds.variant.eq(variant), ['fold','validation_shot_count','validation_goal_count','log_loss','brier_score','accuracy','precision','recall','f1']].copy()
print('Ten validation folds; standard deviation below describes fold variation, not a confidence interval.')
display(fold_view.round(4))
print('Unweighted fold mean and sample standard deviation (ddof=1):')
display(fold_view.drop(columns=['fold']).agg(['mean','std']).round(4))


In [ ]:
row = testing.loc[testing.variant.eq('gaussian_nb_sigmoid')].iloc[0]
print(f"Held-out test: {int(row.shot_count):,} shots, {int(row.goal_count):,} goals; fixed threshold = {row.threshold:.1f}")
print('Probability metrics:')
display(pd.DataFrame([{'Log loss':row.log_loss,'Brier':row.brier_score,
                        'Average precision':row.average_precision,'ROC-AUC':row.roc_auc}]).round(4))
print('Classification metrics and threshold counts:')
display(pd.DataFrame([{'Accuracy':row.accuracy,'Precision':row.precision,'Recall':row.recall,
                        'F1':row.f1,'TN':int(row.tn),'FP':int(row.fp),'FN':int(row.fn),'TP':int(row.tp)}]).round(4))
fig, ax = plt.subplots(figsize=(4.5, 3.5))
matrix = [[int(row.tn), int(row.fp)], [int(row.fn), int(row.tp)]]
im = ax.imshow(matrix, cmap='Blues')
for i in range(2):
    for j in range(2):
        text_color = 'white' if matrix[i][j] > max(max(values) for values in matrix) / 2 else 'black'
        ax.text(j, i, f'{matrix[i][j]:,}', ha='center', va='center', color=text_color, fontsize=12, fontweight='bold')
ax.set_xticks([0,1], ['Non-goal','Goal'])
ax.set_yticks([0,1], ['Non-goal','Goal'])
ax.set_title('Gaussian Naive Bayes: held-out test confusion matrix')
ax.set_xlabel('Predicted class'); ax.set_ylabel('Actual class')
fig.text(.5, .01, 'Test only · threshold 0.5 · StatsBomb Open Data', ha='center', fontsize=8)
fig.tight_layout(rect=[0,.05,1,1]); plt.show()


Capture training, CV, and test evidence as `gaussian_nb_01_training.png`, `gaussian_nb_02_cross_validation.png`, and `gaussian_nb_03_test.png`. Capture the held-out metrics table and confusion matrix separately as `evaluation_gaussian_nb_test_metrics.png` and `evaluation_gaussian_nb_test_confusion.png`. Preserve the visible title and scope label in every image.

### Logistic regression — sigmoid-calibrated family-selected variant

The family-selected variant is the sigmoid-calibrated version chosen by lower pooled training-only OOF log loss than its raw counterpart. These are the three model chapters reported below; the logistic-regression sigmoid variant is the pre-frozen global candidate.


In [ ]:
variant = 'logistic_regression_sigmoid'
print('Logistic regression: same fixed 0.5 decision threshold for hard predictions')
display(scope_table(variant).round(4))
fold_view = folds.loc[folds.variant.eq(variant), ['fold','validation_shot_count','validation_goal_count','log_loss','brier_score','accuracy','precision','recall','f1']].copy()
print('Ten validation folds; standard deviation below describes fold variation, not a confidence interval.')
display(fold_view.round(4))
print('Unweighted fold mean and sample standard deviation (ddof=1):')
display(fold_view.drop(columns=['fold']).agg(['mean','std']).round(4))


In [ ]:
row = testing.loc[testing.variant.eq('logistic_regression_sigmoid')].iloc[0]
print(f"Held-out test: {int(row.shot_count):,} shots, {int(row.goal_count):,} goals; fixed threshold = {row.threshold:.1f}")
print('Probability metrics:')
display(pd.DataFrame([{'Log loss':row.log_loss,'Brier':row.brier_score,
                        'Average precision':row.average_precision,'ROC-AUC':row.roc_auc}]).round(4))
print('Classification metrics and threshold counts:')
display(pd.DataFrame([{'Accuracy':row.accuracy,'Precision':row.precision,'Recall':row.recall,
                        'F1':row.f1,'TN':int(row.tn),'FP':int(row.fp),'FN':int(row.fn),'TP':int(row.tp)}]).round(4))
fig, ax = plt.subplots(figsize=(4.5, 3.5))
matrix = [[int(row.tn), int(row.fp)], [int(row.fn), int(row.tp)]]
im = ax.imshow(matrix, cmap='Blues')
for i in range(2):
    for j in range(2):
        text_color = 'white' if matrix[i][j] > max(max(values) for values in matrix) / 2 else 'black'
        ax.text(j, i, f'{matrix[i][j]:,}', ha='center', va='center', color=text_color, fontsize=12, fontweight='bold')
ax.set_xticks([0,1], ['Non-goal','Goal'])
ax.set_yticks([0,1], ['Non-goal','Goal'])
ax.set_title('Logistic regression: held-out test confusion matrix')
ax.set_xlabel('Predicted class'); ax.set_ylabel('Actual class')
fig.text(.5, .01, 'Test only · threshold 0.5 · StatsBomb Open Data', ha='center', fontsize=8)
fig.tight_layout(rect=[0,.05,1,1]); plt.show()


Capture training, CV, and test evidence as `logistic_01_training.png`, `logistic_02_cross_validation.png`, and `logistic_03_test.png`. Capture the held-out metrics table and confusion matrix separately as `evaluation_logistic_test_metrics.png` and `evaluation_logistic_test_confusion.png`. Preserve the visible title and scope label in every image.

## Submission reminder

The notebook provides screenshot-ready saved outputs, but screenshots are genuine only after the student captures the visible VS Code/Jupyter interface. Save those original images in `reports/screenshots`, insert them into the report at the mapped chapters, and rerender the DOCX for final layout review.
